# 06 — Masked data

**Workload:** Masking invalid sensor readings, reductions, filling, and compressed selections.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import sys

# rnp is the Rust engine's numpy-compatible package: one import swap and
# everything below is ordinary NumPy code.
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "shim"))

import rnp as np

probe = np.array(0)
print("rnp version:", np.__version__)
print(f"engine: {type(probe).__module__}.{type(probe).__name__}")
assert type(probe).__module__ == "_rnp"

rnp version: 2.5.2
engine: _rnp.ndarray


## Mask invalid readings

Represent missing sensor values with `nan`, then convert them to a masked array.

In [2]:
readings = np.array([
    [18.0, 45.0, 1012.0],
    [19.5, np.nan, 1011.0],
    [np.nan, 48.0, 1010.0],
    [21.0, 52.0, np.nan],
])
masked = np.ma.masked_invalid(readings)
mask = np.ma.getmaskarray(masked)
valid_counts = masked.count(axis=0)
print("mask:\n", mask)
print("valid counts:", valid_counts)

mask:
 [[False False False]
 [False  True False]
 [ True False False]
 [False False  True]]
valid counts: [3 3 3]


## Reduce, impute, and select

Compute column means, fill missing entries, and compress a conditional mask.

In [3]:
column_means = masked.mean(axis=0)
imputed = masked.filled(column_means)
hot_and_valid = np.ma.masked_where(masked[:, 0] < 20.0, masked[:, 0])
hot_values = hot_and_valid.compressed()
print("column means:", np.round(column_means.filled(np.nan), 4))
print("imputed readings:\n", imputed)
print("hot valid values:", hot_values)

column means: [  19.5      48.3333 1011.    ]
imputed readings:
 [[  18.           45.         1012.        ]
 [  19.5          48.33333333 1011.        ]
 [  19.5          48.         1010.        ]
 [  21.           52.         1011.        ]]
hot valid values: [21.]


## Verify the result

In [4]:
assert np.array_equal(valid_counts, [3, 3, 3])
assert np.allclose(column_means.filled(np.nan), [19.5, 145.0 / 3.0, 1011.0], rtol=0.0, atol=1e-12)
assert np.array_equal(hot_values, [21.0])
print("PASS — all masked-data assertions passed.")

PASS — all masked-data assertions passed.
